# Task 1

In [11]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from IPython.display import clear_output

plt.ion()

%matplotlib inline

# if GPU is to be used
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [12]:
from model import DQN

In [ ]:
# BATCH_SIZE is the number of transitions sampled from the replay buffer
# GAMMA is the discount factor as mentioned in the previous section
# EPS_START is the starting value of epsilon
# EPS_END is the final value of epsilon
# EPS_DECAY controls the rate of exponential decay of epsilon, higher means a slower decay
# TAU is the update rate of the target network
# LR is the learning rate of the ``AdamW`` optimizer

BATCH_SIZE = 128
GAMMA = 0.99
TAU = 0.005
LR = 3e-4


# Get number of actions from gym action space
n_actions = env.action_space.n
# Get the number of state observations
state, info = env.reset()
n_observations = len(state)

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=LR, amsgrad=True)


steps_done = 0


def select_action(state):
    global steps_done
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1
    if sample > eps_threshold:
        with torch.no_grad():
            # t.max(1) will return the largest column value of each row.
            # second column on max result is index of where max element was
            # found, so we pick action with the larger expected reward.
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)


episode_durations = []

In [ ]:
def compute_loss(
    net: DQN,
    obs: np.ndarray,
    action: int,
    reward: float,
    next_obs: np.ndarray,
    done: bool,
    gamma: float,
    device: torch.device,
) -> torch.Tensor:
    """
    Compute the one-step DQN loss for a *single* transition.

    Loss = ( Q(s,a) − [ r + γ · max_a' Q(s',a') · (1−done) ] )²

    No target network: both Q(s,a) and Q(s',a') come from `net`.
    """
    # --- predicted Q-value for the taken action ---
    q_values = net(preprocess_obs(obs, device))          # (1, n_actions)
    q_sa = q_values[0, action]                          # scalar

    # --- Bellman target (no gradient through target) ---
    with torch.no_grad():
        q_next = net(preprocess_obs(next_obs, device))   # (1, n_actions)
        max_q_next = q_next.max(dim=1).values[0]        # scalar
        target = reward + gamma * max_q_next * (1.0 - float(done))

    loss = F.mse_loss(q_sa, target)
    return loss

In [ ]:
def train_step(q, q_target, optimizer):
    s, a, r, s_, done = memory.sample(batch_size)
    q_out = q(s).gather(1, a)
    max_q_prime = q_target(s_).max(1)[0].unsqueeze(1)
    target = r + gamma * max_q_prime * (1 - done)
    loss = nn.MSELoss()(q_out, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

preprocess_obs transforms obs:
- (H, W, C) to (1, C, H, W) if a single frame is passed
- (N, H, W, C) to (N, C, H, W) if multiple frames are passed, where N = number of frames

In [18]:
import numpy as np

def preprocess_obs(obs):
    obs = torch.tensor(obs.astype(np.float32) / 255.0, device=device)
    if obs.ndim == 3: # add batch dimension = 1 if a single frame is passed
        obs = obs.unsqueeze(0)
    return obs.permute(0, 3, 1, 2) # (N, C, H, W)

Loss = (r + γ * max(Q(s', a')) - Q(s, a))²

In [ ]:
from space_race_env import SpaceRaceEnv
from agent import Agent

N_EPISODES = 100
DIFFICULTY = 0
GAMMA = 0.99
LR = 1e-4
BASE_SEED = 42
net = DQN(n_actions=2).to(device)

def train(n_episodes = N_EPISODES, difficulty = DIFFICULTY):
    agent = Agent()
    optimizer = optim.Adam(net.parameters(), lr=LR)

    episode_scores   = []   # score at end of each training episode
    episode_losses   = []   # mean loss per training episode
    eval_episodes    = []   # x-axis for eval curve
    eval_mean_scores = []   # mean greedy score at eval checkpoints

    episode_rewards = []
    reward_history = []
    for ep in range(n_episodes):
        env = SpaceRaceEnv(difficulty=difficulty, round_time_seconds=60, ticks_per_second=10, obs_mode="rgb", include_semantic_info=False)
        obs, _ = env.reset(seed=BASE_SEED + ep)
        obs = preprocess_obs(obs)
        done = False
        total_reward = 0
        while not done:
            action = agent.select_action(obs)
            action = int(np.clip(int(action), 0, 1))
            next_obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

            next_obs = preprocess_obs(next_obs)
            q_next = net(next_obs)
            max_q_next = q_next.max(dim=1).values[0]
            loss = ((reward + GAMMA*max_q_next) - net(obs)[0, action]) ** 2
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            obs = next_obs

        env.close()

        reward_history.append(total_reward)
        if ep % 5 == 0:
        # Save episode-reward pair
            episode_rewards.append((ep, total_reward))

    return net #episode_scores, episode_losses, eval_episodes, eval_mean_scores

# Train model
print("Training basic DQN (no replay, no target network)")
#net, ep_scores, ep_losses, eval_eps, eval_scores = train()
net = train()

In [ ]:
from space_race_env import SpaceRaceEnv
from agent import Agent

N_EPISODES = 100
DIFFICULTY = 0
GAMMA = 0.99
LR = 1e-4
BASE_SEED = 42
net = DQN(n_actions=2).to(device)

def train(n_episodes = N_EPISODES, difficulty = DIFFICULTY):
    optimizer = optim.Adam(net.parameters(), lr=LR)

    episode_scores   = []   # score at end of each training episode
    episode_losses   = []   # mean loss per training episode
    eval_episodes    = []   # x-axis for eval curve
    eval_mean_scores = []   # mean greedy score at eval checkpoints

    episode_rewards = []
    reward_history = []
    for ep in range(n_episodes):
        env = SpaceRaceEnv(difficulty=difficulty, round_time_seconds=60, ticks_per_second=10, obs_mode="rgb", include_semantic_info=False)
        obs, _ = env.reset(seed=BASE_SEED + ep)
        obs = preprocess_obs(obs)
        done = False
        total_reward = 0
        while not done:
            with torch.no_grad():
                q_values = net(obs)
                action = q_values.argmax().item()

            next_obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

            next_obs = preprocess_obs(next_obs)
            q_next = net(next_obs)
            max_q_next = q_next.max(dim=1).values[0]
            loss = ((reward + GAMMA*max_q_next*(1-int(done))) - net(obs)[0, action]) ** 2
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            obs = next_obs

        env.close()
        print(total_reward)
        reward_history.append(total_reward)
        if ep % 5 == 0:
        # Save episode-reward pair
            episode_rewards.append((ep, total_reward))

    torch.save(net.state_dict(), "../model.pt")

    return net #episode_scores, episode_losses, eval_episodes, eval_mean_scores

# Train model
print("Training basic DQN (no replay, no target network)")
#net, ep_scores, ep_losses, eval_eps, eval_scores = train()
net = train()